In [22]:
import pandas as pd

df = pd.read_csv('/content/spam.csv', encoding='latin-1')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully!
Shape: (5572, 5)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [23]:
# Keep only the useful columns if they exist, otherwise assume we already processed them
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    # Rename columns
    df.columns = ['label', 'message']

# Remove missing and duplicate messages
df = df.dropna()
df = df.drop_duplicates()

# Convert labels into numbers if they aren't already
if df['label'].dtype == object or df['label'].dtype == str:
    df['label'] = df['label'].map({
        'ham': 0,
        'spam': 1
    })
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)

print("Cleaned dataset shape:", df.shape)
display(df.head())
print("\nSpam and non-spam counts:")
print(df['label'].value_counts())

Cleaned dataset shape: (5169, 2)


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."



Spam and non-spam counts:
label
0    4516
1     653
Name: count, dtype: int64


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Text converted successfully!")
print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

Text converted successfully!
Training messages: 4135
Testing messages: 1034


In [25]:
# Create and train the model
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Email Spam Detection model trained successfully!")

Email Spam Detection model trained successfully!


In [26]:
# Predict test messages
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Not Spam', 'Spam']
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 96.62 %

Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.96      1.00      0.98       903
        Spam       0.99      0.74      0.85       131

    accuracy                           0.97      1034
   macro avg       0.98      0.87      0.91      1034
weighted avg       0.97      0.97      0.96      1034


Confusion Matrix:
[[902   1]
 [ 34  97]]


In [30]:
message = input("Enter an email or message: ")

message_tfidf = vectorizer.transform([message])
prediction = model.predict(message_tfidf)[0]

if prediction == 1:
    print("Result: SPAM")
else:
    print("Result: NOT SPAM")

Enter an email or message: Congratulations! You have won a free prize. Claim your reward now!
Result: SPAM


In [29]:
import pickle

with open('spam_detection_model.pkl', 'wb') as f:
    pickle.dump(model, f)

    with open('tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)

        print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [31]:
from google.colab import files

files.download('/content/spam_detection_model.pkl')
files.download('/content/tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>